In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
BASE_PATH = "/content/drive/MyDrive/multimodal_segmentation_qa"

DATA_PATH = f"{BASE_PATH}/data/unified"
MODEL_SAVE_PATH = f"{BASE_PATH}/model_colab.pth"

In [4]:
!pip install torch torchvision opencv-python tqdm

In [5]:
import json, cv2, torch, os
from torch.utils.data import Dataset

class UnifiedDataset(Dataset):
    def __init__(self, metadata_path, img_size=384):
        with open(metadata_path, 'r') as f:
            self.data = json.load(f)

        self.img_size = img_size
        print("Samples:", len(self.data))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        image = cv2.imread(item["image"])
        mask = cv2.imread(item["mask"], 0)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        image = cv2.resize(image, (self.img_size, self.img_size))
        mask = cv2.resize(mask, (self.img_size, self.img_size))

        image = image / 255.0
        mask = mask / 255.0

        prompt_value = 1.0 if item["prompt"] == "segment crack" else 0.0

        h, w, _ = image.shape
        prompt_channel = torch.full((h, w, 1), prompt_value)

        image = torch.tensor(image, dtype=torch.float32)
        image = torch.cat([image, prompt_channel], dim=2)
        image = image.permute(2, 0, 1)

        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return image, mask

In [6]:
import torch.nn as nn
import torchvision.models as models

class ImprovedModel(nn.Module):
    def __init__(self):
        super().__init__()

        base = models.resnet18(weights="DEFAULT")

        self.conv1 = nn.Conv2d(4, 64, 7, 2, 3, bias=False)

        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.up1 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.up4 = nn.ConvTranspose2d(64, 32, 2, 2)

        self.final = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        x = self.up4(x)

        return self.final(x)

In [7]:
import torch.nn.functional as F

def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    return 1 - (2. * intersection) / (pred.sum() + target.sum() + 1e-6)

In [9]:
import torch
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
from tqdm import tqdm

BATCH_SIZE = 6
EPOCHS = 25
LR = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

dataset = UnifiedDataset(f"{DATA_PATH}/metadata_fixed.json")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=0)

model = ImprovedModel().to(device)

bce = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]).to(device))

optimizer = optim.Adam(model.parameters(), lr=LR)

def dice_score(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.3).float()
    intersection = (pred * target).sum()
    return (2. * intersection) / (pred.sum() + target.sum() + 1e-6)

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for images, masks in loop:
        images, masks = images.to(device), masks.to(device)

        outputs = model(images)
        outputs = F.interpolate(outputs, size=masks.shape[-2:], mode='bilinear')

        loss = bce(outputs, masks) + dice_loss(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    model.eval()
    dice_total = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)

            outputs = model(images)
            outputs = F.interpolate(outputs, size=masks.shape[-2:], mode='bilinear')

            dice_total += dice_score(outputs, masks).item()

    print(f"\nEpoch {epoch+1}: Dice = {dice_total/len(val_loader):.4f}")

torch.save(model.state_dict(), MODEL_SAVE_PATH)
print("Improved model saved!")

Using: cuda
Samples: 2044


Epoch 1/25: 100%|██████████| 273/273 [00:54<00:00,  5.04it/s, loss=1.49]



Epoch 1: Dice = 0.2798


Epoch 2/25: 100%|██████████| 273/273 [00:53<00:00,  5.14it/s, loss=1.53]



Epoch 2: Dice = 0.3221


...


Epoch 23: Dice = 0.6749


Epoch 24/25: 100%|██████████| 273/273 [00:52<00:00,  5.23it/s, loss=0.704]



Epoch 24: Dice = 0.6587


Epoch 25/25: 100%|██████████| 273/273 [00:55<00:00,  4.88it/s, loss=0.134]



Epoch 25: Dice = 0.6786
Improved model saved!
